# Лекција 10 - AI агенти у продукцији

У овој лекцији ћете научити **продукцијске обрасце** за AI агенте користећи Microsoft Agent Framework (Python). Обухватамо:

- **Надгледање** — додавање мерења времена и логовања у интеракције агента
- **Евалуација** — коришћење евалуатора агента за оцењивање квалитета одговора
- **Управљање трошковима** — стратегије за оптимизацију токена и избор модела

Сценарио је **агент за путовања** који помаже корисницима да планирају путовања, са надгледањем и евалуацијом као додатним слојевима.


## Постављање


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Разматрања за продукцију

Прелазак AI агената из прототипа у продукцију захтева пажљиву посвећеност трима стубовима:

1. **Observability** — Потребна вам је видљивост у то шта агент ради, колико времена троши и које алате позива. Без праћења и логовања, отклањање грешака у продукцији је готово немогуће.

2. **Evaluation** — Аутоматизоване контроле квалитета обезбеђују да одговори агента остану тачни, потпуни и корисни током времена. Агент за евалуацију може оцењивати одговоре према дефинисаним критеријумима.

3. **Cost Management** — Коришћење токена директно утиче на трошкове. Стратегије као што су оптимизација упита, избор модела и кеширање помажу да се трошкови држе под контролом без жртвовања квалитета.


## Изградња опажљивог агента

Дефинишемо алате за путовања и умотавамо позив агента у мерење времена тако да можемо да пратимо латенцију. У продукцији бисте интегрисали OpenTelemetry или сличан бекенд за праћење.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Обрасци евалуације

Уобичајени продукциони образац је коришћење другог агента као **оценитеља**. Оценитељ оцењује одговор примарног агента у односу на унапред дефинисане критеријуме као што су потпуност, тачност и корисност.

Ово омогућава:
- Аутоматизоване контроле квалитета пре него што одговори стигну до корисника
- Откривање регресија када се упити или модели промене
- Континуирано праћење перформанси агента током времена


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегије управљања трошковима

Контрола трошкова је кључна за AI агенте у продукцији. Ево главних стратегија:

| Стратегија | Опис |
|---|---|
| **Оптимизација промпта** | Држите системске инструкције концизним. Уклоните сувишни контекст да бисте смањили улазне токене. |
| **Избор модела** | Користите мање, јефтиније моделе (нпр. GPT-4o-mini) за једноставне задатке као што су класификација или извлачење, а веће моделе оставите за сложено расуђивање. |
| **Кеширање** | Кеширајте резултате алата и учестале упите да бисте избегли сувишне API позиве. |
| **Буџети токена** | Поставите ограничења `max_tokens` да бисте спречили неочекивано дуге одговоре. |
| **Груписање** | Групишите више корисничких упита у један API позив где је то могуће. |

У пракси, слојевити приступ добро функционише: усмерите једноставне захтеве ка брзом, јефтином моделу и само сложене упите проследите способнијем.


## Резиме

У овој лекцији сте научили како да:

1. **Додати набљудивост** у интеракције агента кроз мерење времена и логовање, постављајући темеље за трасирање и праћење.
2. **Проценити одговоре агента** аутоматски користећи агента-оцењивача који оцењује потпуност, тачност и корисност.
3. **Управљати трошковима** кроз оптимизацију упита, избор модела, кеширање и буџете токена.

Ови производни обрасци помажу да ваши агенти вештачке интелигенције буду поуздани, мерљиви и исплативи на великој скали.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Одрицање одговорности:
Овај документ је преведен помоћу услуге за превођење засноване на вештачкој интелигенцији Co‑op Translator (https://github.com/Azure/co-op-translator). Иако се трудимо да обезбедимо тачност, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Првобитни документ на његовом оригиналном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионалан превод од стране човека. Не прихватамо одговорност за било какве неспоразуме или погрешне тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
